In [1]:
import psycopg2
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT 
from psycopg2.extras import execute_batch

import pandas as pd

In [2]:
with open('pwd.txt', 'w') as file:
    file.write(input('Enter the password: '))
with open('pwd.txt', 'r') as file:
    pwd = file.read()
conn = psycopg2.connect(
        dbname='postgres',
        user='postgres',
        password=pwd,
        host='localhost',
        port='5432')

Enter the password:  111207Geo


In [3]:
cur = conn.cursor()
conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
cur.execute("CREATE DATABASE poject;")

In [4]:
# ILONA -------------------------------------------------
df = pd.read_csv('flight_data_.csv')

mapping = {'int64' : 'INTEGER',
        'str' : 'VARCHAR(100)',
        'object': 'VARCHAR(100)',
        'float64' : 'NUMERIC(10, 2)',
        'bool' : 'BOOLEAN'}
postgre_dtypes = df.dtypes.reset_index(drop=False).iloc[:, 1].astype(str)
postgre_dtypes = postgre_dtypes.apply(lambda x: mapping[x])

cols = ', '.join([f'"{col}"' for col in df.columns])
columns = ''
for c, t in zip(df.columns, postgre_dtypes):
    columns += f'"{c}" {t}, '
columns = columns.strip(', ')
vals = ', '.join(['%s'] * len(df.columns))

cur.execute(f'''DROP TABLE IF EXISTS data; 
            CREATE TABLE data ({columns});''')
cur.executemany(f'INSERT INTO data ({cols}) VALUES ({vals});', [row for row in df.itertuples(index=False)])

FileNotFoundError: [Errno 2] No such file or directory: 'flight_data_.csv'

In [ ]:
'''importing csv file to pandas DataFrame
creating dictionary to convert pandas datatypes to SQL datatypes
taking list of pandas datatypes
converting to SQL datatypes by mapping
making a one-row srtring of list of columns to save place in SQL query
making empty string to fullfill it with column names and their datatypes
fullfilling it as requierd for SQL
just string with %s that means that we insert such amount of vals
creating main table with our dataset in SQL
just insert and iter by rows for making list of tuples of rows
'''

In [ ]:
# GLEB-------------------------------------------------
cur.execute('''ALTER TABLE data 
ALTER COLUMN "Departure Date" TYPE TIMESTAMP 
USING TO_TIMESTAMP("Departure Date", 'YYYY-MM-DD HH24:MI:SS');''')

cur.execute("""
DROP TABLE IF EXISTS data_clean;

CREATE TABLE data_clean AS
SELECT 
    *,
    COALESCE("Delay Minutes", 0) AS "Delay Minutes Clean"
FROM data;
""")

cur.execute('''
DROP TABLE IF EXISTS flights_schedule;
CREATE TABLE flights_schedule AS
SELECT
    ROW_NUMBER() OVER (ORDER BY "Route", "Departure Date") AS "Flight ID",
    "Route",
    "Departure Date",
    MAX("Delay Minutes Clean") AS "Delay Minutes Clean",
    COUNT(*) AS "Passenger Count",
    AVG("Ticket Price") AS "Avg Ticket Price"
FROM data_clean
GROUP BY "Route", "Departure Date";
''')

cur.execute('''
DROP TABLE IF EXISTS customers;
CREATE TABLE customers AS
SELECT DISTINCT ON ("Customer ID")
    "Customer ID",
    "Name",
    COUNT(*) OVER(PARTITION BY "Customer ID") AS "Flights count",
    "Frequent Flyer Status",
    "Loyalty Points"
FROM data_clean
ORDER BY "Customer ID", "Departure Date" DESC;
''')

In [ ]:
'''Convert the flight date to date type
Create a “cleaned table” in case there is no value in the delay column (to use COALESCE)
Create a flight schedule table
Create a passenger table
'''

In [ ]:
# ILONA-------------------------------------------------
cur.execute('''
DROP TABLE IF EXISTS tickets;
CREATE TABLE tickets AS
SELECT 
    f."Flight ID",           
    d."Customer ID",         
    d."Booking Class",       
    d."Ticket Price",       
    d."Competitor Price",    
    d."Demand",              
    d."Profitability"     
FROM data_clean d
JOIN flights_schedule f 
    ON f."Route" = d."Route" 
    AND f."Departure Date" = d."Departure Date";''')

df_tickets = pd.read_sql("""
SELECT *
FROM tickets
LIMIT 10;
""", conn)

df_tickets.head()

In [ ]:
cur.execute('''
DROP TABLE IF EXISTS tickets_left;
CREATE TABLE tickets_left AS
SELECT 
    f."Flight ID",
    d."Customer ID",
    d."Booking Class",
    d."Ticket Price"
FROM data_clean d
LEFT JOIN flights_schedule f 
    ON f."Route" = d."Route" 
    AND f."Departure Date" = d."Departure Date";''')
df_ticketsleft = pd.read_sql("""
SELECT *
FROM tickets_left
LIMIT 10;
""", conn)

df_tickets.head()

In [ ]:
cur.execute('''
DROP TABLE IF EXISTS flights_with_passengers;
CREATE TABLE flights_with_passengers AS
SELECT 
    f."Flight ID",
    f."Route",
    f."Departure Date",
    f."Passenger Count",
    d."Customer ID"
FROM data_clean d
RIGHT JOIN flights_schedule f 
    ON f."Route" = d."Route" 
    AND f."Departure Date" = d."Departure Date";''')
df_flights_with_passengers = pd.read_sql("""
SELECT *
FROM flights_with_passengers
LIMIT 10;
""", conn)

df_flights_with_passengers.head()

In [ ]:
cur.execute('''
DROP TABLE IF EXISTS full_join;
CREATE TABLE full_join AS
SELECT 
    f."Flight ID",
    d."Customer ID",
    d."Booking Class",
    COALESCE(f."Route", d."Route") AS "Route"
FROM data_clean d
FULL OUTER JOIN flights_schedule f 
    ON f."Route" = d."Route" 
    AND f."Departure Date" = d."Departure Date";''')
df_full = pd.read_sql("""
SELECT *
FROM full_join
LIMIT 10;
""", conn)

df_full.head()

In [ ]:
'''this section is about working with join types
actually it makes no sense as our schedule table is made from data table, so it's just subset of it
and all JOINs return the same result (this is just a syntax demonstration)
'join' is 'inner join' by default - only passengers who have a matching flight in the schedule (kinda tickets)
left join - all passengers from data_clean, flight fields may be NULL (if flight is missing)
right join - all flights from the schedule, passenger fields may be NULL (if no passengers on a flight)
fulL outer join - all records from both tables, fields may be NULL on either side
'''

In [ ]:
# GLASHA -------------------------------------------------
cur.execute('''
DROP TABLE IF EXISTS rolling_3days;
CREATE TABLE rolling_3days AS
WITH rolling_calc AS (
    SELECT 
        "Flight ID",
        "Route",
        "Departure Date",
        "Passenger Count",
        COUNT(*) OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS window_size,
        ROUND(AVG("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling Avg 3 Days",
        ROUND(STDDEV("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling StdDev 3 Days"
    FROM flights_schedule
)
SELECT 
    "Flight ID",
    "Route",
    "Departure Date",
    "Passenger Count",
    CASE WHEN window_size = 3 THEN "Rolling Avg 3 Days" ELSE NULL END AS "Rolling Avg 3 Days",
    CASE WHEN window_size = 3 THEN "Rolling StdDev 3 Days" ELSE NULL END AS "Rolling StdDev 3 Days",
    CASE 
        WHEN window_size = 3 AND "Rolling Avg 3 Days" > LAG("Rolling Avg 3 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'INCREASING'
        WHEN window_size = 3 AND "Rolling Avg 3 Days" < LAG("Rolling Avg 3 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'DECLINING'
        WHEN window_size = 3 THEN 'STABLE'
        ELSE NULL
    END AS "Demand Trend"
FROM rolling_calc
WHERE window_size = 3;''')
df_rolling = pd.read_sql("""
SELECT *
FROM rolling_3days
LIMIT 10;
""", conn)

df_rolling.head()

In [ ]:
'''this calculates moving averages and standard deviations to analyze demand trends
counting how many rows are in the current rolling window to exclude where window is not 3
calculating 3-day rolling average of passenger count
calculating 3-day rolling standard deviation to measure demand volatility
determining demand trend by comparing current rolling avg with previous one
the same for 6-day, 9-day
'''

In [ ]:
cur.execute('''
DROP TABLE IF EXISTS rolling_6days;
CREATE TABLE rolling_6days AS
WITH rolling_calc AS (
    SELECT 
        "Flight ID",
        "Route",
        "Departure Date",
        "Passenger Count",
        COUNT(*) OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ) AS window_size,
        ROUND(AVG("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling Avg 6 Days",
        ROUND(STDDEV("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling StdDev 6 Days"
    FROM flights_schedule
)
SELECT 
    "Flight ID",
    "Route",
    "Departure Date",
    "Passenger Count",
    CASE WHEN window_size = 6 THEN "Rolling Avg 6 Days" ELSE NULL END AS "Rolling Avg 6 Days",
    CASE WHEN window_size = 6 THEN "Rolling StdDev 6 Days" ELSE NULL END AS "Rolling StdDev 6 Days",
    CASE 
        WHEN window_size = 6 AND "Rolling Avg 6 Days" > LAG("Rolling Avg 6 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'INCREASING'
        WHEN window_size = 6 AND "Rolling Avg 6 Days" < LAG("Rolling Avg 6 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'DECLINING'
        WHEN window_size = 6 THEN 'STABLE'
        ELSE NULL
    END AS "Demand Trend"
FROM rolling_calc
WHERE window_size = 6;''')
df_rolling_6 = pd.read_sql("""
SELECT *
FROM rolling_6days
LIMIT 10;
""", conn)

df_rolling_6.head()

In [ ]:
cur.execute('''
DROP TABLE IF EXISTS rolling_9days;
CREATE TABLE rolling_9days AS
WITH rolling_calc AS (
    SELECT 
        "Flight ID",
        "Route",
        "Departure Date",
        "Passenger Count",
        COUNT(*) OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 8 PRECEDING AND CURRENT ROW
        ) AS window_size,
        ROUND(AVG("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 8 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling Avg 9 Days",
        ROUND(STDDEV("Passenger Count") OVER (
            PARTITION BY "Route" 
            ORDER BY "Departure Date" 
            ROWS BETWEEN 8 PRECEDING AND CURRENT ROW
        ), 2) AS "Rolling StdDev 9 Days"
    FROM flights_schedule
)
SELECT 
    "Flight ID",
    "Route",
    "Departure Date",
    "Passenger Count",
    CASE WHEN window_size = 9 THEN "Rolling Avg 9 Days" ELSE NULL END AS "Rolling Avg 9 Days",
    CASE WHEN window_size = 9 THEN "Rolling StdDev 9 Days" ELSE NULL END AS "Rolling StdDev 9 Days",
    CASE 
        WHEN window_size = 9 AND "Rolling Avg 9 Days" > LAG("Rolling Avg 9 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'INCREASING'
        WHEN window_size = 9 AND "Rolling Avg 9 Days" < LAG("Rolling Avg 9 Days") OVER (PARTITION BY "Route" ORDER BY "Departure Date") THEN 'DECLINING'
        WHEN window_size = 9 THEN 'STABLE'
        ELSE NULL
    END AS "Demand Trend"
FROM rolling_calc
WHERE window_size = 9;''')
df_rolling_9 = pd.read_sql("""
SELECT *
FROM rolling_9days
LIMIT 10;
""", conn)

df_rolling_9.head()

In [ ]:
# KATE ---------------------------------------
cur.execute("""
ALTER TABLE flights_schedule
ALTER COLUMN "Departure Date" TYPE TIMESTAMP
USING "Departure Date"::timestamp;
""")
cur.execute('ALTER TABLE flights_schedule ADD COLUMN "Delay Category" VARCHAR(30);')
cur.execute('ALTER TABLE flights_schedule ADD COLUMN "On Time Flag" INTEGER;')
cur.execute('''UPDATE flights_schedule
SET 
    "Delay Category" = CASE
        WHEN "Delay Minutes Clean" < 15  THEN 'on-time'
        WHEN "Delay Minutes Clean" BETWEEN 15 AND 60 THEN 'minor delay'
        WHEN "Delay Minutes Clean" BETWEEN 60 AND 180 THEN 'major delay'
        WHEN "Delay Minutes Clean" > 180 THEN 'severe delay'
        ELSE 'cancellation' 
    END,
    "On Time Flag" = CASE WHEN "Delay Minutes Clean" < 15 THEN 1 ELSE 0 END;''')

cur.execute('''CREATE TABLE rolling_on_time_percentage AS
SELECT 
    "Flight ID",
    "Route",
    "Departure Date",
    "Delay Category",
    "On Time Flag",
    ROUND(
        AVG("On Time Flag") OVER (
            PARTITION BY "Route"
            ORDER BY "Departure Date"
            RANGE BETWEEN INTERVAL '7 days' PRECEDING AND CURRENT ROW
        ) * 100, 2
    ) AS "Rolling On-Time % (last 7 days)"
FROM flights_schedule
ORDER BY "Route", "Departure Date";''')

In [ ]:
'''creating a rolling on-time performance table using 7-day calendar windows
calculating the percentage of on-time flights (delay < 15 minutes) for each route
over the last 7 days up to and including the current flight date'''

In [ ]:
df_on_time = pd.read_sql("""
SELECT *
FROM rolling_on_time_percentage
LIMIT 10;
""", conn)

df_on_time.head(9)

In [ ]:
cur.close()
conn.close()